# BeeAI — Pipeline de Treino (Kaggle)

Execute as células em ordem. GPU T4 gratuita — ative em **Settings > Accelerator > GPU T4**.

Antes de começar, adicione o secret `GITHUB_TOKEN` em **Add-ons > Secrets** se o repositório for privado.

In [ ]:
# Célula 1 — Verificar GPU
import subprocess, torch

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("GPU não detectada. Ative em Settings > Accelerator > GPU T4.")

In [ ]:
# Célula 2 — Configuração
# Edite os valores abaixo antes de executar

# URL do repositório (público ou privado)
GITHUB_REPO_URL = "https://github.com/SEU_USUARIO/identificadordeabelha"

# Nome da versão do modelo (vira o nome do diretório de artefatos)
MODEL_VERSION = "dinov2_small_mlp_v001"

# Encoder: dinov2-small (~80 MB, mais rápido) ou dinov2-base (~350 MB, mais preciso)
ENCODER_NAME = "facebook/dinov2-small"

# Imagens por espécie baixadas do GBIF
# 100 = ~8 min de download, bom para validação
# 300 = ~25 min de download, melhor generalização
IMAGES_PER_SPECIES = 150

print(f"Encoder:  {ENCODER_NAME}")
print(f"Versão:   {MODEL_VERSION}")
print(f"Imagens:  {IMAGES_PER_SPECIES} por espécie")

In [ ]:
# Célula 3 — Carregar tokens via Kaggle Secrets
# Adicione em: Add-ons > Secrets > Add new secret
#   GITHUB_TOKEN  — obrigatório se o repo for privado
#   HF_TOKEN      — opcional (dinov2-small/base são públicos)

GITHUB_TOKEN = ""
HF_TOKEN = ""

try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    try:
        GITHUB_TOKEN = _s.get_secret("GITHUB_TOKEN")
        print("GITHUB_TOKEN carregado.")
    except Exception:
        print("GITHUB_TOKEN não encontrado — usando sem token (só funciona para repos públicos).")
    try:
        HF_TOKEN = _s.get_secret("HF_TOKEN")
        print("HF_TOKEN carregado.")
    except Exception:
        print("HF_TOKEN não encontrado — ok para modelos públicos.")
except Exception:
    print("Kaggle Secrets não disponível.")

In [ ]:
# Célula 4 — Clonar repositório
from pathlib import Path

REPO_DIR = Path("/kaggle/working/identificadordeabelha")

if REPO_DIR.exists():
    print("Repositório já existe.")
else:
    clone_url = GITHUB_REPO_URL
    if GITHUB_TOKEN:
        clone_url = clone_url.replace("https://", f"https://{GITHUB_TOKEN}@")
    !git clone {clone_url} /kaggle/working/identificadordeabelha

print(f"Conteúdo: {[p.name for p in sorted(REPO_DIR.iterdir())]}")

In [ ]:
# Célula 5 — Instalar dependências
%cd /kaggle/working/identificadordeabelha/backend
!pip install -e "." -q
print("Dependências instaladas.")

In [ ]:
# Célula 6 — Criar diretórios e configurar ambiente
# O config.py usa REPO_ROOT/data e REPO_ROOT/artifacts por padrão
# Só precisamos criar os diretórios fisicamente
import os
from pathlib import Path

REPO_DIR = Path("/kaggle/working/identificadordeabelha")

for d in [
    REPO_DIR / "data/raw/images/gbif_demo",
    REPO_DIR / "data/raw/metadata",
    REPO_DIR / "data/interim/embeddings",
    REPO_DIR / "data/interim/metadata",
    REPO_DIR / "data/processed",
    REPO_DIR / "data/db",
    REPO_DIR / "data/uploads",
    REPO_DIR / "artifacts/models",
]:
    d.mkdir(parents=True, exist_ok=True)

env_content = ""
if HF_TOKEN:
    env_content = f"HF_TOKEN={HF_TOKEN}\n"
    os.environ["HF_TOKEN"] = HF_TOKEN
(REPO_DIR / ".env").write_text(env_content)

print("Pronto. Caminhos padrão:")
print(f"  Dataset:    {REPO_DIR}/data/raw")
print(f"  Artefatos:  {REPO_DIR}/artifacts/models")
print(f"  Banco:      {REPO_DIR}/data/db/app.sqlite3")

In [ ]:
# Célula 7 — Baixar dataset do GBIF
# Espécies padrão: Augochloropsis metallica, A. callichroa, A. ignita
# Para outras espécies: adicione --species "Nome Um" "Nome Dois" no comando abaixo
%cd /kaggle/working/identificadordeabelha/backend

REPO = "/kaggle/working/identificadordeabelha"
print(f"Baixando {IMAGES_PER_SPECIES} imagens por espécie do GBIF...\n")

!python scripts/download_gbif_demo_dataset.py \
  --images-per-species {IMAGES_PER_SPECIES} \
  --raw-images-dir {REPO}/data/raw/images/gbif_demo \
  --raw-catalog {REPO}/data/raw/metadata/raw_catalog.csv \
  --species-seed {REPO}/data/raw/metadata/species_seed.demo.csv \
  --sources {REPO}/data/raw/metadata/gbif_demo_sources.csv \
  --allow-all-licenses

In [ ]:
# Célula 8 — Verificar dataset
import pandas as pd

catalog = pd.read_csv("/kaggle/working/identificadordeabelha/data/raw/metadata/raw_catalog.csv")
print(f"Total de imagens: {len(catalog)}")
print("\nPor espécie:")
print(catalog["species_code"].value_counts().to_string())

if len(catalog) < 9:
    print("\nAVISO: Dataset muito pequeno. O GBIF pode ter retornado menos imagens que o esperado.")

In [ ]:
# Célula 9 — Pipeline de treino
# Etapas: limpeza → split → embeddings (GPU) → MLP → avaliação → empacotamento
%cd /kaggle/working/identificadordeabelha/backend

REPO = "/kaggle/working/identificadordeabelha"

print(f"Iniciando pipeline: {MODEL_VERSION}")
print(f"Encoder:            {ENCODER_NAME}")
print("-" * 60)

!python scripts/run_training_pipeline.py \
  --raw-catalog {REPO}/data/raw/metadata/raw_catalog.csv \
  --clean-catalog {REPO}/data/interim/metadata/clean_catalog.csv \
  --split-manifest {REPO}/data/interim/metadata/split_manifest.json \
  --processed-root {REPO}/data/processed \
  --embeddings-dir {REPO}/data/interim/embeddings \
  --artifacts-root {REPO}/artifacts/models \
  --version {MODEL_VERSION} \
  --encoder-name {ENCODER_NAME}

In [ ]:
# Célula 10 — Resultados
import json, matplotlib.pyplot as plt, matplotlib.image as mpimg
from pathlib import Path

REPO = Path("/kaggle/working/identificadordeabelha")
run_dir = REPO / "artifacts/models/_runs" / MODEL_VERSION

metrics_path = run_dir / "metrics.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    print("=== Métricas ===")
    for k, v in metrics.items():
        bar = "=" * int(v * 30) if isinstance(v, float) else ""
        print(f"  {k:<25} {v:.4f if isinstance(v, float) else v}  {bar}")
else:
    print(f"metrics.json não encontrado em {run_dir}")

confusion_path = run_dir / "confusion_matrix.png"
if confusion_path.exists():
    print("\n=== Matriz de Confusão ===")
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(mpimg.imread(str(confusion_path)))
    ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Célula 11 — Empacotar artefatos para download
import shutil
from pathlib import Path

REPO = Path("/kaggle/working/identificadordeabelha")
packaged_dir = REPO / "artifacts/models" / MODEL_VERSION

print(f"Artefatos em {packaged_dir}:")
for p in sorted(packaged_dir.rglob("*")):
    if p.is_file():
        print(f"  {str(p.relative_to(packaged_dir)):<40} {p.stat().st_size / 1024:>8.1f} KB")

# Zipar — o arquivo vai para /kaggle/working/ e fica disponível no Output
zip_out = f"/kaggle/working/{MODEL_VERSION}"
shutil.make_archive(zip_out, "zip", str(packaged_dir.parent), MODEL_VERSION)

zip_size = Path(zip_out + ".zip").stat().st_size / 1e6
print(f"\nArquivo: {MODEL_VERSION}.zip ({zip_size:.1f} MB)")
print("Download: painel direito > Output > Download output files")